# Figure 09 -- weak scaling

Loads `bench/results/multigpu/weak_scaling.json`, produced by `bench/multigpu/weak_scaling.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Contention on a shared host is additive and intermittent, so a sweep's own
# median can be inflated at any point -- measured: two of three invocations of the
# six-device weak point read 455 and 399 ms against a true 229. The MINIMUM across
# independent invocations is the estimator section 5 quotes, so the figures use it
# too, or figure and text disagree at exactly the contended points.
def _min_over_invocations(stem):
    # -> ({ndev: record}, config), keeping per ndev the lowest-median invocation.
    best, cfg = {}, None
    for suffix in ("", "_rep1", "_rep2"):
        try:
            art = jsonio.read_result("multigpu/" + stem + suffix + ".json")
        except FileNotFoundError:
            continue
        cfg = art["config"]
        for r in art["data"]["records"]:
            if not r.get("valid"):
                continue
            d = r["ndev"]
            if d not in best or r["median_s"] < best[d]["median_s"]:
                best[d] = r
    if not best:
        raise SystemExit("no valid " + stem + " records in any invocation")
    return best, cfg

best, cfg = _min_over_invocations("weak_scaling")
recs = [best[d] for d in sorted(best)]

ok = list(recs)
bad = []  # invalid points already dropped by _min_over_invocations
if not ok:
    raise SystemExit(
        "weak_scaling.json has no valid points: the per-device load overflows "
        "even at the smallest device count."
    )
ok.sort(key=lambda r: r["ndev"])

ndev = [r["ndev"] for r in ok]
thru = [r["throughput_particles_per_s"] for r in ok]
t = [r["median_s"] for r in ok]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2)

# Throughput. Perfect weak scaling is a straight line through the origin: each
# added device adds its own share of work and its own share of throughput.
axes[0].plot(ndev, thru, marker=style.MARKERS[0], color=style.entity_color("thru", 0),
             label="measured")
ideal = [thru[0] * (d / ndev[0]) for d in ndev]
axes[0].plot(ndev, ideal, linestyle=":", linewidth=0.8, color="0.5", label="ideal")
axes[0].set_xlabel("devices")
axes[0].set_ylabel("throughput [particles s$^{-1}$]")
axes[0].set_title("throughput at fixed load per device", fontsize=8)
style.finish(axes[0], legend=True, legend_kwargs={"loc": "best", "fontsize": 6.2})

# Time per evaluation. Flat is perfect; the rise is communication plus whatever
# padding the capacity retries added.
axes[1].plot(ndev, t, marker=style.MARKERS[1], color=style.entity_color("wall", 1))
axes[1].axhline(t[0], linestyle=":", linewidth=0.8, color="0.5")
axes[1].set_xlabel("devices")
axes[1].set_ylabel("time per force evaluation [s]")
axes[1].set_title("cost per evaluation (flat is ideal)", fontsize=8)
style.finish(axes[1], legend=False)

note = jsonio.config_caption(cfg, ["order", "leaf_size", "precision", "device"])
note += f"\n{cfg.get('per_device_n_fixed')} particles/device"
if bad:
    note += f"  |  {len(bad)} point(s) dropped (overflow)"
style.annotate_config(axes[0], note)
style.save(fig, FIG_DIR / "fig09_weak_scaling.pdf")

for r in sorted(recs, key=lambda r: r["ndev"]):
    flag = "" if r.get("valid") else "   INVALID"
    print(f"ndev={r['ndev']:<3d} n={r.get('n')} median={r.get('median_s')} "
          f"throughput={r.get('throughput_particles_per_s')}{flag}")


## Caption

Weak scaling: the particle count per device is held fixed and devices are added,
so the ideal throughput is linear in device count and the ideal cost per
evaluation is flat. This is the load-bearing scaling measurement for this
implementation, because the per-device particle count is what the traversal
buffers are sized against; holding it fixed is the regime the code is built to
run in.